In [9]:
from typing import List, Dict, Any
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END


In [10]:
import getpass
import os
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_deepseek import ChatDeepSeek

deepseek_api_key = "sk-f46c7e2053764299ace45b6e3d98de77"
base_url = "https://api.deepseek.com"

# model = ChatOpenAI(model="gpt-4o")
if not os.getenv("DEEPSEEK_API_KEY"):
    os.environ["DEEPSEEK_API_KEY"] = deepseek_api_key

llm = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    # other params...
)


# @tool
# def magic_function(input: int) -> int:
#     """Applies a magic function to an input."""
#     return input + 2


# tools = [magic_function]


# query = "what is the value of magic_function(77)?"


In [11]:
file_path = './titanic_cleaned.csv'

In [12]:
import pandas as pd
df = pd.read_csv(file_path)
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,C92,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,C92,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,C92,S


In [14]:
import pandas as pd
from langgraph.graph import StateGraph, END
from typing import TypedDict, Any, List
from langchain.chat_models import ChatOpenAI
from langchain.schema import SystemMessage, HumanMessage

# 定义共享状态
class GraphState(TypedDict):
    step: str
    data: Any  # 用于保存Agent上一步结果
    prompt_data: str  # 给LLM的真实数据描述文本

# 全局历史记录列表
history: List[dict] = []

# 读取你的真实数据
titanic_df = pd.read_csv(file_path)  # 请替换为实际路径

def describe_dataframe(df: pd.DataFrame) -> str:
    desc = []
    desc.append(f"数据集包含 {df.shape[0]} 行，{df.shape[1]} 列。")
    desc.append("字段信息如下：")
    for col in df.columns:
        dtype = df[col].dtype
        missing = df[col].isna().sum()
        desc.append(f"- {col}，类型：{dtype}，缺失值数量：{missing}。")
        if pd.api.types.is_numeric_dtype(dtype):
            stats = df[col].describe()
            desc.append(
                f" 统计信息 - 均值: {stats['mean']:.2f}, 标准差: {stats['std']:.2f}, "
                f"最小值: {stats['min']}, 最大值: {stats['max']}"
            )
    return "\n".join(desc)

def prepare_prompt_data(df: pd.DataFrame, sample_size=50) -> str:
    desc = describe_dataframe(df)
    sample = df.sample(min(sample_size, len(df)), random_state=42)
    sample_str = sample.to_csv(index=False, sep='\t')
    return f"{desc}\n\n数据样本（共{len(sample)}条）：\n{sample_str}"

def call_llm(role: str, task_description: str, data_description: str) -> str:
    messages = [
        SystemMessage(content=f"你是一名{role}，需要独立完成数据分析任务，并且只能基于我给你的数据和需求来做推理。"),
        HumanMessage(content=f"任务描述：{task_description}\n数据描述：{data_description}\n请直接给出最终结论或结果，使用简洁的中文表述。")
    ]
    return llm(messages).content

# Agent定义，调用时用状态里的 prompt_data
class DataSummaryAgent:
    def __call__(self, state: GraphState):
        print("[DataSummaryAgent] 正在计算数据摘要...")
        prompt_task = "计算数值列的均值、方差、最大值、最小值，并找出潜在异常值。"
        prompt_data = state["prompt_data"]
        result = call_llm("数据分析师", prompt_task, prompt_data)
        print("[DataSummaryAgent] 分析结果：", result)
        history.append({"step": "summary", "result": result})
        return {"step": "missing_value", "data": result, "prompt_data": prompt_data}

class MissingValueAgent:
    def __call__(self, state: GraphState):
        print("[MissingValueAgent] 正在填补缺失值...")
        prompt_task = "找出数据集中缺失值的列，并使用均值填充这些列的缺失值。"
        prompt_data = state["prompt_data"]
        result = call_llm("数据清洗专家", prompt_task, prompt_data)
        print("[MissingValueAgent] 填充说明：", result)
        history.append({"step": "missing_value", "result": result})
        return {"step": "visualization", "data": result, "prompt_data": prompt_data}

class VisualizationAgent:
    def __call__(self, state: GraphState):
        print("[VisualizationAgent] 正在绘制 Survived 列的分布图...")
        prompt_task = "分析 Survived 列的分布情况，统计生存与未生存的比例，并说明可能的原因。"
        prompt_data = state["prompt_data"]
        result = call_llm("数据可视化专家", prompt_task, prompt_data)
        print("[VisualizationAgent] 分析说明：", result)
        history.append({"step": "visualization", "result": result})
        return {"step": "model_training", "data": result, "prompt_data": prompt_data}

class ModelTrainingAgent:
    def __call__(self, state: GraphState):
        print("[ModelTrainingAgent] 正在训练模型并进行预测...")
        prompt_task = "使用 sklearn 训练一个模型，根据乘客特征预测其是否能生存，并评估模型准确率。"
        prompt_data = state["prompt_data"]
        result = call_llm("机器学习工程师", prompt_task, prompt_data)
        print("[ModelTrainingAgent] 模型说明：", result)
        history.append({"step": "model_training", "result": result})
        return {"step": END, "data": result, "prompt_data": prompt_data}

# 构建 langgraph 流程
workflow = StateGraph(GraphState)
workflow.add_node("summary", DataSummaryAgent())
workflow.add_node("missing_value", MissingValueAgent())
workflow.add_node("visualization", VisualizationAgent())
workflow.add_node("model_training", ModelTrainingAgent())

workflow.set_entry_point("summary")
workflow.add_edge("summary", "missing_value")
workflow.add_edge("missing_value", "visualization")
workflow.add_edge("visualization", "model_training")
workflow.add_edge("model_training", END)

app = workflow.compile()

# 准备 prompt_data
prompt_data = prepare_prompt_data(titanic_df)

# 启动执行，初始state包含prompt_data
app.invoke({"step": "summary", "data": None, "prompt_data": prompt_data})

# 执行完毕后，history中保存了所有步骤结果
print("\n=== 中间过程历史记录 ===")
for record in history:
    print(f"步骤 {record['step']} 结果:\n{record['result']}\n")


[DataSummaryAgent] 正在计算数据摘要...


/tmp/ipykernel_1130571/2783822975.py:46: LangChainDeprecationWarning: The method `BaseChatModel.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  return llm(messages).content


[DataSummaryAgent] 分析结果： 数值列统计结果：

1. PassengerId：
   - 均值：446.00
   - 方差：66229.02
   - 最大值：891.0
   - 最小值：1.0
   - 异常值：无（ID连续分布）

2. Survived：
   - 均值：0.38
   - 方差：0.24
   - 最大值：1.0
   - 最小值：0.0
   - 异常值：无（二元分类变量）

3. Pclass：
   - 均值：2.31
   - 方差：0.71
   - 最大值：3.0
   - 最小值：1.0
   - 异常值：无（1-3等舱分类）

4. Age：
   - 均值：29.70
   - 方差：169.00
   - 最大值：80.0
   - 最小值：0.42
   - 异常值：0.42（婴儿年龄）和80.0（高龄乘客）

5. SibSp：
   - 均值：0.52
   - 方差：1.21
   - 最大值：8.0
   - 最小值：0.0
   - 异常值：8.0（异常多的兄弟姐妹/配偶）

6. Parch：
   - 均值：0.38
   - 方差：0.66
   - 最大值：6.0
   - 最小值：0.0
   - 异常值：6.0（异常多的父母/子女）

7. Fare：
   - 均值：32.20
   - 方差：2469.10
   - 最大值：512.33
   - 最小值：0.0
   - 异常值：512.33（极端高票价）和0.0（免费票）

注：Age列中29.69911764705882为缺失值的填充值，实际应视为缺失数据而非异常值。
[MissingValueAgent] 正在填补缺失值...
[MissingValueAgent] 填充说明： 根据数据分析结果：
1. 数据集中所有列均无缺失值（各列缺失值数量均为0）
2. 因此不需要进行均值填充处理

结论：无需进行缺失值填充操作，因为所有字段均完整无缺失。
[VisualizationAgent] 正在绘制 Survived 列的分布图...
[VisualizationAgent] 分析说明： 生存比例分析结果：
1. 生存率：38%（342人）
2. 死亡率：62%（549人）

可能原因：
1. 性别因素：样本中女性

In [15]:
history

[{'step': 'summary',
  'result': '数值列统计结果：\n\n1. PassengerId：\n   - 均值：446.00\n   - 方差：66229.02\n   - 最大值：891.0\n   - 最小值：1.0\n   - 异常值：无（ID连续分布）\n\n2. Survived：\n   - 均值：0.38\n   - 方差：0.24\n   - 最大值：1.0\n   - 最小值：0.0\n   - 异常值：无（二元分类变量）\n\n3. Pclass：\n   - 均值：2.31\n   - 方差：0.71\n   - 最大值：3.0\n   - 最小值：1.0\n   - 异常值：无（1-3等舱分类）\n\n4. Age：\n   - 均值：29.70\n   - 方差：169.00\n   - 最大值：80.0\n   - 最小值：0.42\n   - 异常值：0.42（婴儿年龄）和80.0（高龄乘客）\n\n5. SibSp：\n   - 均值：0.52\n   - 方差：1.21\n   - 最大值：8.0\n   - 最小值：0.0\n   - 异常值：8.0（异常多的兄弟姐妹/配偶）\n\n6. Parch：\n   - 均值：0.38\n   - 方差：0.66\n   - 最大值：6.0\n   - 最小值：0.0\n   - 异常值：6.0（异常多的父母/子女）\n\n7. Fare：\n   - 均值：32.20\n   - 方差：2469.10\n   - 最大值：512.33\n   - 最小值：0.0\n   - 异常值：512.33（极端高票价）和0.0（免费票）\n\n注：Age列中29.69911764705882为缺失值的填充值，实际应视为缺失数据而非异常值。'},
 {'step': 'missing_value',
  'result': '根据数据分析结果：\n1. 数据集中所有列均无缺失值（各列缺失值数量均为0）\n2. 因此不需要进行均值填充处理\n\n结论：无需进行缺失值填充操作，因为所有字段均完整无缺失。'},
 {'step': 'visualization',
  'result': '生存比例分析结果：\n1. 生存率：38%（342人）\n2. 死亡率：62%（54